In [1]:
import xarray as xr
from meteodatalab import ogd_api
from meteodatalab.operators import regrid
from rasterio.crs import CRS
from datetime import timedelta
from earthkit.data import config
import numpy as np

config.set("cache-policy", "temporary")

# --- Grid definition ---
extent = (-0.817, 18.183, 41.183, 51.183)
nx, ny = 429, 295
destination = regrid.RegularGrid(CRS.from_epsg(4326), nx, ny, *extent)

# --- Fetch + regrid all lead times ---
das = []
for h in range(0, 34):
    req = ogd_api.Request(
        collection="ogd-forecasting-icon-ch1",
        variable="TOT_PREC",
        reference_datetime="latest",
        perturbed=True,
        horizon=timedelta(hours=h),
    )
    da_h = ogd_api.get_from_ogd(req)
    da_h_geo = regrid.iconremap(da_h, destination)
    das.append(da_h_geo)
    print(f"✓ +{h:02d}h")

# --- Concatenate ---
da_all = xr.concat(das, dim="lead_time")
da_all = da_all.assign_coords(lead_time=[timedelta(hours=h) for h in range(34)])

# --- Strip non-serializable attrs ---
def clean_attrs(attrs):
    valid_types = (str, int, float, bytes, list, tuple)
    return {k: v for k, v in attrs.items() if isinstance(v, valid_types)}

da_all.attrs = clean_attrs(da_all.attrs)
for coord in da_all.coords:
    da_all[coord].attrs = clean_attrs(da_all[coord].attrs)

# --- Compute hourly rain ---
mean_precip = da_all.mean("eps").squeeze("ref_time")        # (lead_time, y, x)
hourly_rain = mean_precip.diff("lead_time")                 # ← this line was missing!
hourly_rain = hourly_rain.drop_vars("valid_time", errors="ignore")
hourly_rain.values = np.where(hourly_rain.values < 0.01, 0.0, np.round(hourly_rain.values, 2))
hourly_rain.attrs = {"long_name": "Hourly precipitation (ensemble mean)", "units": "mm/m2"}

# --- Save ---
ds = xr.Dataset({
    "TOT_PREC": da_all,
    "hourly_rain": hourly_rain,
})
ds.to_netcdf("total_0-33.nc")


try:
    ds.close()
except: pass

print("Saved!")

icon-ch1-eps-202604060600-0-tot_prec-perturb.grib2?AWSAccessKeyId=13GC1T4NT2CY1N0L92YC&Signature=OlGCG02SiS1Yu…

horizontal_constants_icon-ch1-eps.grib2?AWSAccessKeyId=13GC1T4NT2CY1N0L92YC&Signature=sr3jR0nv8lgzh%2FMykV6UO1…

✓ +00h


icon-ch1-eps-202604060600-1-tot_prec-perturb.grib2?AWSAccessKeyId=13GC1T4NT2CY1N0L92YC&Signature=hxuqIa%2BcV92…

✓ +01h


...

✓ +31h


icon-ch1-eps-202604060600-32-tot_prec-perturb.grib2?AWSAccessKeyId=13GC1T4NT2CY1N0L92YC&Signature=zoNYZk8d22LB…

✓ +32h


icon-ch1-eps-202604060600-33-tot_prec-perturb.grib2?AWSAccessKeyId=13GC1T4NT2CY1N0L92YC&Signature=zAhJLb0BY0fK…

✓ +33h
Saved!


In [2]:
ds = xr.open_dataset("icon_ch1_TOT_PREC_all_lead_times.nc")
da_all = ds["TOT_PREC"]
hourly_rain = ds["hourly_rain"]

def sel_latlon(da, lat, lon):
    dist = np.sqrt((da.lat - lat)**2 + (da.lon - lon)**2)
    iy, ix = np.unravel_index(dist.values.argmin(), dist.shape)
    return da.isel(y=iy, x=ix)

# --- Basel at +17h ---
rain_basel   = sel_latlon(hourly_rain, lat=47.56, lon=7.59)
tot_basel    = sel_latlon(da_all,      lat=47.56, lon=7.59)

rain_17h     = float(rain_basel.isel(lead_time=16))          # diff index 16 = between +16h and +17h
prob_17h     = float((
    (tot_basel.isel(lead_time=17) - tot_basel.isel(lead_time=16))
    .mean("eps") > 0.1
) * 100)  # simplified — use per-member diff for proper probability:
prob_17h     = float(((tot_basel.sel(eps=slice(1,10)).isel(lead_time=17) - 
                        tot_basel.sel(eps=slice(1,10)).isel(lead_time=16)) > 0.1).mean("eps") * 100)

print(f"Basel   +17h → rainfall: {rain_17h:.2f} mm/m²  |  probability: {prob_17h:.0f}%")

# --- Zürich at +30h ---
rain_zurich  = sel_latlon(hourly_rain, lat=47.37, lon=8.54)
tot_zurich   = sel_latlon(da_all,      lat=47.37, lon=8.54)

rain_30h     = float(rain_zurich.isel(lead_time=29))         # diff index 29 = between +29h and +30h
prob_30h     = float(((tot_zurich.sel(eps=slice(1,10)).isel(lead_time=30) - 
                        tot_zurich.sel(eps=slice(1,10)).isel(lead_time=29)) > 0.1).mean("eps") * 100)

print(f"Zürich  +30h → rainfall: {rain_30h:.2f} mm/m²  |  probability: {prob_30h:.0f}%")

Basel   +17h → rainfall: 0.00 mm/m²  |  probability: 0%
Zürich  +30h → rainfall: 0.00 mm/m²  |  probability: 0%


In [3]:
# --- Find all cells with heavy rain (>5mm in any single hour) in next 33h ---
# hourly_rain shape: (lead_time=33, y=295, x=429)

THRESHOLD = 3.5  # mm — adjust to taste (2.0 = moderate, 5.0 = heavy, 10.0 = very heavy)

# Boolean mask: any hour exceeds threshold → shape (y, x)
heavy_mask = (hourly_rain > THRESHOLD).any(dim="lead_time")

# Get y/x indices of all matching cells
iy_all, ix_all = np.where(heavy_mask.values)

print(f"Found {len(iy_all)} cells with >{THRESHOLD}mm rain in any single hour\n")

for iy, ix in zip(iy_all, ix_all):
    lat = float(hourly_rain.lat.isel(y=iy, x=ix))
    lon = float(hourly_rain.lon.isel(y=iy, x=ix))
    
    # Which hours and how much?
    cell_rain = hourly_rain.isel(y=iy, x=ix)
    heavy_hours = [(h+1, float(v)) for h, v in enumerate(cell_rain.values) if v > THRESHOLD]
    hours_str = ", ".join([f"+{h:02d}h={v:.1f}mm" for h, v in heavy_hours])
    
    print(f"  lat={lat:.2f} lon={lon:.2f}  →  {hours_str}")

Found 13 cells with >3.5mm rain in any single hour

  lat=43.70 lon=2.82  →  +10h=3.8mm
  lat=45.77 lon=4.29  →  +11h=3.6mm
  lat=45.77 lon=4.33  →  +11h=3.6mm
  lat=45.77 lon=4.38  →  +11h=3.5mm
  lat=45.81 lon=4.38  →  +11h=4.3mm
  lat=47.65 lon=14.10  →  +17h=4.0mm
  lat=47.68 lon=13.30  →  +16h=3.8mm
  lat=47.68 lon=13.34  →  +16h=3.8mm
  lat=47.68 lon=14.10  →  +17h=3.7mm
  lat=47.68 lon=15.16  →  +18h=3.9mm
  lat=47.71 lon=14.10  →  +17h=5.0mm
  lat=47.71 lon=14.14  →  +17h=3.9mm
  lat=47.71 lon=14.99  →  +18h=4.2mm


In [4]:
print(hourly_rain.isel(y=iy, x=ix))

<xarray.DataArray 'hourly_rain' (lead_time: 34)> Size: 272B
array([ nan, 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ,
       0.11, 0.  , 0.02, 0.75, 0.64, 4.22, 0.94, 0.34, 0.15, 0.13, 0.33, 0.16,
       0.12, 0.06, 0.03, 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ])
Coordinates:
  * lead_time  (lead_time) timedelta64[ns] 272B 00:00:00 ... 1 days 09:00:00
    lon        float64 8B 14.99
    lat        float64 8B 47.71
Attributes:
    long_name:  Hourly precipitation (ensemble mean)
    units:      mm/m2
